# Notebook 09: breaking the best detector, part 3: collective anomalies (water plant)

**Class 03_03, part 3 of 3.** A **collective** anomaly is a group of values that is abnormal together although each
one looks fine: a sensor that repeats a stale value is in range, but it no longer moves with the others. The
plant of notebook 06 has eight sensors driven by two hidden processes (demand and heat), so the sensors are
correlated and the detectors of class 03_02 learn exactly that correlation.

1. **Pick** the best detector for the three events of notebook 06 (spike, freeze, drift).
2. **Break** it twice: a *coordinated lie* that keeps the correlations intact, and a *slow drift* that the per-point
   threshold never sees.
3. **Defend**: sensors the attacker cannot reach (guards), and CUSUM.

**Threat model.** The attacker controls the readings of $m$ of the 8 sensors (a compromised gateway, not the
plant itself) and knows the family of the detector (linear residuals) but not the threshold. Goal: make the *level*
sensor read $\Delta$ standard deviations too high while no alarm is raised.

In [ ]:
import os
import sys
from itertools import combinations

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import polars as pl
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import average_precision_score

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
import deeplib as dl
import robustlib as rl
from aasplot import BLUE, GOLD, GREEN, GREY, RED

an.table_config()
ap.setup("03")
Xn, _, _ = dl.plant(seed=0)
Xa, ya, events = dl.plant(seed=1, attacks=True)
mu, sd = Xn.mean(0), Xn.std(0)
Xn, Xa = (Xn - mu) / sd, (Xa - mu) / sd
LEVEL = dl.SENSORS.index("level")

## A. Pick the best detector

The detectors of notebook 06 on the attacked run, one threshold rule (99.5% quantile of the scores on the normal
run). The best is the one with the highest AUPRC over the attacked run.

In [ ]:
def resid_models(X):
    return [LinearRegression().fit(np.delete(X, i, 1), X[:, i]) for i in range(X.shape[1])]


def resid(X, ms):
    return np.max([np.abs(X[:, i] - ms[i].predict(np.delete(X, i, 1))) for i in range(X.shape[1])], 0)


pca = PCA(2).fit(Xn)
ms = resid_models(Xn)
iso = an.IForest().fit(Xn)
ae = dl.AE(latent=2, widths=(16,), epochs=30, seed=0).fit(Xn)
W = 48
cae = dl.fit_window_ae(dl.conv_ae(W, 8), dl.windows(Xn, W), epochs=30, seed=0)
DET = {
    "per-sensor max|z|": lambda X: np.abs(X).max(1),
    "PCA recon (space)": lambda X: ((X - pca.inverse_transform(pca.transform(X))) ** 2).sum(1),
    "predict from others (space)": lambda X: resid(X, ms),
    "Isolation Forest (space)": iso.score,
    "dense AE (space)": ae.score,
    "conv AE (space+time)": lambda X: dl.window_to_point(dl.window_error(cae, dl.windows(X, W)), W, len(X)),
}
rows = []
for k, f in DET.items():
    s, t = f(Xa), np.quantile(f(Xn), 0.995)
    rows.append({"detector": k, "AUPRC": average_precision_score(ya, s), **dl.events_found(s > t, events),
                 "false alarms": int(((s > t) & (ya == 0)).sum())})  # fmt: skip
pick = pl.DataFrame(rows).sort("AUPRC", descending=True)
BEST = pick["detector"][0]
print(pick)
print("best:", BEST)

## B. Break it (1): a coordinated lie that keeps the correlations

Both space detectors measure a **residual** that is linear in the readings: PCA keeps the part of $\mathbf{x}$ outside
the 2-D normal subspace, $\mathbf{r}=(I-P)\mathbf{x}$; "predict from the others" uses
$r_i=x_i-\mathbf{w}_i^\top\mathbf{x}_{-i}$. Writing $\mathbf{r}=A\mathbf{x}$, an attacker who controls the set $C$ of
sensors adds $\boldsymbol\delta$ with $\delta_{\text{level}}=\Delta$ and the other entries in $C$ chosen to
minimise $\lVert A\boldsymbol\delta\rVert$: a least-squares problem, no search, no queries. The more sensors
the attacker owns, the closer the lie sits to the normal subspace.

In [ ]:
P_ = pca.components_.T @ pca.components_
A_PCA = np.eye(8) - P_
A_REG = np.stack(
    [np.insert(-ms[i].coef_, i, 1.0) for i in range(8)]
)  # r = A_REG @ x (+ intercepts ~ 0 on centred data)
SCORE = {
    "PCA recon (space)": (A_PCA, lambda R: (R**2).sum(1)),
    "predict from others (space)": (A_REG, lambda R: np.abs(R).max(1)),
}
target = BEST if BEST in SCORE else "PCA recon (space)"  # the linear residual detectors can be attacked in closed form
A, agg = SCORE[target]
t_det = np.quantile(agg((Xn - Xn.mean(0)) @ A.T), 0.995)
clean = (Xa[ya == 0] - Xa[ya == 0].mean(0))[:2000]


def lie(A, C, delta, s=LEVEL):
    """The shift vector: delta on sensor s, least-squares helpers on the other compromised sensors C."""
    others = [c for c in C if c != s]
    d = np.zeros(A.shape[1])
    d[s] = delta
    if others:
        d[others] = np.linalg.lstsq(A[:, others], -A[:, s] * delta, rcond=None)[0]
    return d


def best_helpers(A, m, delta, pool):
    """The m-1 helper sensors (with the level sensor: m compromised) that leave the smallest residual; exhaustive,
    at most C(7, 3) = 35 candidates."""
    return min(([LEVEL, *h] for h in combinations([j for j in pool if j != LEVEL], m - 1)),
               key=lambda C: np.linalg.norm(A @ lie(A, C, delta)))  # fmt: skip


rows = []
for delta in [1, 2, 3]:
    for m in range(1, 9):
        C = best_helpers(A, m, delta, range(8))
        alarm = float((agg((clean + lie(A, C, delta)) @ A.T) > t_det).mean())
        rows.append(
            {
                "delta (sd)": delta,
                "sensors owned": m,
                "alarm rate": alarm,
                "sensors": ", ".join(dl.SENSORS[c] for c in C),
            }
        )
lies = pl.DataFrame(rows)
print(f"attacked detector: {target}; clean alarm rate {float((agg(clean @ A.T) > t_det).mean()):.3f}")
print(lies.pivot(on="delta (sd)", index="sensors owned", values="alarm rate").sort("sensors owned"))

In [ ]:
def coordinated_lie(lies) -> None:
    """Alarm rate of the lie on the level sensor against the number of sensors the attacker owns, for three sizes."""
    f, ax = ap.fig("full", 1.5)
    for delta, col, mk in ((1, GREEN, "o"), (2, GOLD, "s"), (3, RED, "^")):
        d = lies.filter(pl.col("delta (sd)") == delta).sort("sensors owned")
        ax.plot(d["sensors owned"], d["alarm rate"], color=col, marker=mk, ms=3.5, label=f"level lies by {delta} sd")
    ax.axhline(0.005, color=GREY, ls=":", lw=1)
    ax.text(8, 0.04, "clean false-alarm rate", color=GREY, fontsize=6.5, ha="right")
    ax.set(xlabel="sensors the attacker controls (of 8)", ylabel="alarm rate during the lie", ylim=(-0.02, 1.05))
    ax.set_xticks(range(1, 9))
    ax.legend(loc="lower left", fontsize=6.8)
    ap.save(f, "coordinated_lie")


coordinated_lie(lies)

**Reading.** PCA over the sensors is the best detector (AUPRC 0.60, all three events, 14 false alarms), and it is
exact about what it checks: that the readings lie on the 2-D normal subspace. A level reading 2 sd too high is
detected at once (alarm rate 1.0) while the attacker owns 1 to 4 sensors; with 5 of the 8 the least-squares helpers
bring the residual under the threshold (alarm rate 0.02), and with 6 or more the lie is invisible, even for 3 sd. No
query and no search were needed: the detector's own geometry gives the attack. Per-point checks cannot see this;
only a sensor the attacker cannot write can.

## C. Break it (2): a slow drift under a per-point threshold

The pressure sensor drifts slowly. A per-point detector (the residual of pressure predicted from the other sensors)
compares each instant with a fixed threshold; a drift that adds a little each step can stay under it for hours.
**CUSUM** (Page 1954) accumulates small deviations: $S_t=\max(0,\,S_{t-1}+(r_t-\mu)/\sigma-k)$, alarm when $S_t>h$.

In [ ]:
P = 7  # pressure


def hours_to_alarm(seed: int, slope_per_day: float) -> tuple[float | None, float | None]:
    """Fit on the first half of a plant run, inject the drift in the second half, hours until the first alarm."""
    X, _, _ = dl.plant(seed=seed)
    half = len(X) // 2
    X = (X - X[:half].mean(0)) / X[:half].std(0)
    reg = LinearRegression().fit(np.delete(X[:half], P, 1), X[:half, P])
    X[half:, P] += slope_per_day * np.arange(len(X) - half) / dl.DAY  # standard deviations per day
    r = X[:, P] - reg.predict(np.delete(X, P, 1))
    mu, sd = r[:half].mean(), r[:half].std()
    c = rl.cusum(r, mu, sd)
    t_point, h = np.abs(r[:half] - mu).max() * 1.1, c[:half].max() * 1.1  # same rule: above anything seen before
    first = lambda m: float(np.flatnonzero(m[half:])[0] / 12) if m[half:].any() else None  # noqa: E731  hours
    return first(np.abs(r - mu) > t_point), first(c > h)


rows = []
for slope in [0.1, 0.25, 0.5, 1.0, 2.0]:
    res = [hours_to_alarm(seed, slope) for seed in range(5)]
    med = lambda v: float(np.median([x if x is not None else np.inf for x in v]))  # noqa: E731
    rows.append({"drift (sd/day)": slope, "per-point: median hours": med([a for a, _ in res]),
                 "CUSUM: median hours": med([b for _, b in res]),
                 "per-point: missed (of 5)": sum(a is None for a, _ in res), "CUSUM: missed (of 5)": sum(b is None for _, b in res)})  # fmt: skip
print(pl.DataFrame(rows))

In [ ]:
def slow_drift(rows) -> None:
    """rows: dicts with `drift (sd/day)`, `per-point: median hours`, `CUSUM: median hours`, `per-point: missed (of 5)`."""
    x = np.arange(len(rows))
    f, ax = ap.fig("full", 1.25)
    pp = [r["per-point: median hours"] for r in rows]
    cu = [r["CUSUM: median hours"] for r in rows]
    for off, v, col, lab in ((-0.19, pp, BLUE, "per-point threshold"), (0.19, cu, RED, "CUSUM")):
        bars = ax.bar(
            x + off, np.array(v) - 1, 0.36, bottom=1, color=col, label=lab, lw=0
        )  # log axis: bars grow from 1
        for b, val in zip(bars, v):
            ax.text(b.get_x() + b.get_width() / 2, val * 1.08, f"{val:.1f}", ha="center", fontsize=7)
    for xi, r in zip(x, rows):
        if r["per-point: missed (of 5)"]:
            ax.text(
                xi - 0.19, 1.2, f"{r['per-point: missed (of 5)']} of 5\nnever", ha="center", fontsize=6, color="white"
            )
    ax.set_xticks(x, [f"{r['drift (sd/day)']:g}" for r in rows])
    ax.set(
        yscale="log",
        xlabel="drift of the pressure sensor (standard deviations per day)",
        ylabel="hours to alarm (median)",
        ylim=(1, 160),
    )
    ax.legend(loc="upper right")
    ax.grid(axis="x", visible=False)
    ap.save(f, "slow_drift")


slow_drift(rows)

Same false-alarm rule for both (above anything seen in the first half). The slower the drift, the longer a per-point
threshold waits (2.5 days at 0.1 sd/day, and one run never fires); CUSUM fires within 2 to 10 hours because it sums
the evidence.
The same idea defends against slow exfiltration (many small sessions) and against the boiling frog of part 1:
test **accumulated** change, not only each point.

## D. Defend it (1): sensors the attacker cannot reach

The lie works because every sensor it touches can be adjusted. A **guard** is a reading the attacker cannot write:
a sensor on a separate network, a second instrument, a value the operator reads by hand. We ask: for each set of $g$
guards, how many of the *other* sensors must the attacker own for a 2-sd lie to stay under the threshold (alarm rate
below 5%)? The search is exhaustive (at most 56 guard sets, 128 attacker sets each).

In [ ]:
def attacker_needs(A, agg, t, guard, delta=2.0, limit=0.05):
    """Fewest sensors (outside the guard set) the attacker must own for an alarm rate below `limit`; None: impossible."""
    free = [j for j in range(8) if j not in guard and j != LEVEL]
    for m in range(1, len(free) + 2):
        for helpers in combinations(free, m - 1):
            C = [LEVEL, *helpers]
            if (agg((clean + lie(A, C, delta)) @ A.T) > t).mean() < limit:
                return m
    return None


rows = []
for g in range(0, 4):
    res = [(attacker_needs(A, agg, t_det, set(G)), G) for G in combinations([j for j in range(8) if j != LEVEL], g)]
    ok = [r for r in res if r[0] is not None]
    worst = max(res, key=lambda r: (r[0] is None, r[0] or 0))  # the guard set that hurts the attacker most
    rows.append({"guards": g, "best guard set": ", ".join(dl.SENSORS[j] for j in worst[1]) or "-",
                 "sensors the attacker must own": worst[0] if worst[0] else "impossible",
                 "guard sets that block it": f"{sum(r[0] is None for r in res)} of {len(res)}"})  # fmt: skip
print(pl.DataFrame(rows))

**Reading.** With no guard the lie needs 5 sensors. One guard (the temperature sensor `temp_a`) raises it to 7 of the 8,
and **two** guards make a 2-sd lie impossible for 17 of the 21 pairs (34 of 35 triples): once the attacker cannot write
two independent readings, the rest cannot be made consistent with them. Which sensors to guard is a design decision
(a separate network, a second instrument) that this exhaustive search can inform. CUSUM (section C) covers the other
attack: it fires within 2 to 10 hours on a drift of 0.1 to 2 sd/day where the per-point threshold waits 5 to 61
hours, but it adds nothing against the lie, which leaves no residual to accumulate.

## E. Defend it (2): accumulate, do not only compare

CUSUM (section C) catches the slow drift; against the coordinated lie it adds nothing, because the lie leaves no
residual to accumulate. The defences are different for different attacks: **guards** for the lie, **accumulation** for
the drift. Together with the other two parts:

| part | best detector | what broke it | what helps | what it costs |
|:--|:--|:--|:--|:--|
| 1 point | 2-layer AE | volume/timing mimicry, a poisoned window, a moved threshold | robust features, vetted retraining, clean validation data | recall on volume-dependent attacks |
| 2 contextual | @@P2@@ | a slow ramp under the seasonal difference, a small sustained shift | two models that fail differently, a frozen reference | false alarms, slower adaptation |
| 3 collective | PCA residual | a lie that follows the correlations; a drift | guard sensors, CUSUM | instruments that the attacker cannot reach |

**No part has a detector that wins on accuracy and robustness at once.** State the threat model, measure the clean
metrics and the robustness together, and say what the defence costs.